# Calculate the metrics from the csv

In [ ]:
import torch
import os
from pathlib import Path

# 1. IMPORT YOUR ACTUAL MODEL ARCHITECTURE
from models.dinov2_builder import DINOv2ReID 

# Import your other project components
from data.dataloader import build_test_loaders
from configs.config import Config
from evaluation_utils import (
    generate_distance_csv,
    calculate_metrics_from_csv
)

In [ ]:
# 1. Setup Configuration
cfg = Config()
cfg.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model_path = "experiments/dinov2_closed_v1/best_model.pth"
cfg.output_dir = Path("csvs/dinov2_closed_v1")
cfg.output_dir.mkdir(parents=True, exist_ok=True)

# 2. Initialize and Load Model
print(f"-> Loading: {best_model_path}")
model = DINOv2ReID() 

checkpoint = torch.load(best_model_path, map_location=cfg.device)
state_dict = checkpoint.get('model', checkpoint.get('state_dict', checkpoint))

# Strip 'module.' prefix if it exists
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)
model.to(cfg.device)
model.eval()

# 3. Build Dataloaders
print("-> Preparing test dataloaders...")
query_loader, gallery_loader = build_test_loaders(cfg)

# 4. Generate and Save Distance CSV
print(f"-> Inference (Chunk Size: {cfg.chunk_size})")
csv_filename = "closed_dist_matrix.csv"

csv_path = generate_distance_csv(
    model, 
    query_loader, 
    gallery_loader, 
    cfg, 
    filename=csv_filename
)

# 5. Calculate Metrics
print("\n" + "="*30)
print("  CLOSED WORLD RESULTS")
print("="*30)
metrics = calculate_metrics_from_csv(csv_path)

print(f"\nDone. CSV: {csv_path}")
print(f"Rank-1: {metrics['Rank-1']:.2%}")